# 02 - Extracción de texto de PDFs

Este notebook realiza la extracción de texto de los documentos PDF registrados en el corpus normativo ambiental.

Entrada principal:

- `data/metadata/corpus_normativo_ambiental.csv`
- PDFs ubicados en `data/raw/`

Salida generada:

- archivos `.txt` por documento en `data/processed/`
- reporte de extracción en `experiments/resultados/`
- versión actualizada del CSV con información de extracción en `data/metadata/`

Este paso prepara el corpus para la siguiente etapa: **chunking estructural**.


## 1. Importación de librerías

In [ ]:
import os
import re
import sys
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

# Instalación / importación de PyMuPDF
try:
    import fitz  # PyMuPDF
except ModuleNotFoundError:
    print("PyMuPDF no está instalado. Instalando...")
    !{sys.executable} -m pip install pymupdf
    import fitz
    print("PyMuPDF instalado correctamente.")

print("Librerías importadas correctamente.")


ImportError: No se encontró PyMuPDF. Install con: pip install pymupdf

## 2. Definición de rutas del proyecto

El notebook puede ejecutarse desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [ ]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    ROOT_DIR = current_dir.parent
else:
    ROOT_DIR = current_dir

DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"
REPORTS_DIR = ROOT_DIR / "experiments" / "resultados"

CSV_PATH = METADATA_DIR / "corpus_normativo_ambiental.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CSV_PATH: {CSV_PATH}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")


## 3. Carga del corpus

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f"Documentos registrados en el corpus: {len(df)}")
df.head()


## 4. Funciones de limpieza y extracción

La extracción se realizará con **PyMuPDF**.  
Se conservarán separadores por página para facilitar después el chunking y la trazabilidad.


In [ ]:
def clean_text(text: str) -> str:
    """Limpieza básica del texto extraído."""
    if not isinstance(text, str):
        return ""

    # Normalizar saltos de línea
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Eliminar espacios excesivos por línea
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]

    # Eliminar múltiples líneas vacías consecutivas
    cleaned_lines = []
    previous_empty = False

    for line in lines:
        is_empty = line == ""
        if is_empty and previous_empty:
            continue
        cleaned_lines.append(line)
        previous_empty = is_empty

    return "\n".join(cleaned_lines).strip()


def extract_text_with_pymupdf(pdf_path: Path) -> dict:
    """Extrae texto de un PDF usando PyMuPDF y devuelve texto + métricas."""
    result = {
        "text": "",
        "pages": 0,
        "pages_with_text": 0,
        "empty_pages": 0,
        "characters": 0,
        "words": 0,
        "status": "ERROR",
        "error": None,
        "method": "PyMuPDF",
    }

    try:
        doc = fitz.open(pdf_path)
        result["pages"] = len(doc)

        page_texts = []

        for page_index, page in enumerate(doc, start=1):
            raw_text = page.get_text("text")
            cleaned_page_text = clean_text(raw_text)

            if cleaned_page_text:
                result["pages_with_text"] += 1
            else:
                result["empty_pages"] += 1

            page_texts.append(f"\n\n===== PÁGINA {page_index} =====\n\n{cleaned_page_text}")

        doc.close()

        full_text = clean_text("\n".join(page_texts))
        result["text"] = full_text
        result["characters"] = len(full_text)
        result["words"] = len(full_text.split())

        if result["characters"] == 0:
            result["status"] = "SIN_TEXTO"
        elif result["pages"] > 0 and result["pages_with_text"] / result["pages"] < 0.7:
            result["status"] = "PARCIAL"
        else:
            result["status"] = "OK"

    except Exception as e:
        result["error"] = str(e)
        result["status"] = "ERROR"

    return result


def infer_texto_extraible(status: str) -> str:
    """Convierte el estado técnico de extracción al valor usado en el CSV."""
    if status == "OK":
        return "Sí"
    if status == "PARCIAL":
        return "Parcial"
    if status == "SIN_TEXTO":
        return "No"
    return "No determinado"


## 5. Prueba rápida con el primer documento

Antes de procesar todo, se prueba con el primer PDF registrado en el corpus.


In [ ]:
first_row = df.iloc[0]
first_pdf = RAW_DIR / first_row["archivo_pdf"]

print(f"Probando extracción con: {first_pdf.name}")

if not first_pdf.exists():
    print(f"No existe el archivo: {first_pdf}")
else:
    test_result = extract_text_with_pymupdf(first_pdf)
    print({
        "status": test_result["status"],
        "pages": test_result["pages"],
        "pages_with_text": test_result["pages_with_text"],
        "characters": test_result["characters"],
        "words": test_result["words"],
        "error": test_result["error"],
    })
    print("\nMuestra de texto extraído:\n")
    print(test_result["text"][:1500])


## 6. Extracción de texto de todos los PDFs

Se procesa cada documento del CSV.  
Por cada PDF se genera un `.txt` en `data/processed/`.


In [ ]:
extraction_results = []
updated_rows = []

for _, row in df.iterrows():
    id_documento = str(row["id_documento"])
    archivo_pdf = str(row["archivo_pdf"])
    pdf_path = RAW_DIR / archivo_pdf

    txt_filename = f"{id_documento}.txt"
    txt_path = PROCESSED_DIR / txt_filename

    if not pdf_path.exists():
        result_record = {
            "id_documento": id_documento,
            "archivo_pdf": archivo_pdf,
            "archivo_txt": txt_filename,
            "status": "PDF_NO_ENCONTRADO",
            "pages": 0,
            "pages_with_text": 0,
            "empty_pages": 0,
            "characters": 0,
            "words": 0,
            "method": "PyMuPDF",
            "error": f"No existe el archivo {pdf_path}",
        }
        extraction_results.append(result_record)

        updated_row = row.to_dict()
        updated_row["archivo_txt"] = txt_filename
        updated_row["estado_extraccion"] = "PDF_NO_ENCONTRADO"
        updated_row["caracteres_extraidos"] = 0
        updated_row["paginas_pdf"] = 0
        updated_row["texto_extraible_detectado"] = "No determinado"
        updated_rows.append(updated_row)
        continue

    extraction = extract_text_with_pymupdf(pdf_path)

    if extraction["status"] in {"OK", "PARCIAL"}:
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(extraction["text"])

    result_record = {
        "id_documento": id_documento,
        "archivo_pdf": archivo_pdf,
        "archivo_txt": txt_filename,
        "status": extraction["status"],
        "pages": extraction["pages"],
        "pages_with_text": extraction["pages_with_text"],
        "empty_pages": extraction["empty_pages"],
        "characters": extraction["characters"],
        "words": extraction["words"],
        "method": extraction["method"],
        "error": extraction["error"],
    }
    extraction_results.append(result_record)

    updated_row = row.to_dict()
    updated_row["archivo_txt"] = txt_filename
    updated_row["estado_extraccion"] = extraction["status"]
    updated_row["caracteres_extraidos"] = extraction["characters"]
    updated_row["paginas_pdf"] = extraction["pages"]
    updated_row["texto_extraible_detectado"] = infer_texto_extraible(extraction["status"])
    updated_rows.append(updated_row)

extraction_df = pd.DataFrame(extraction_results)
updated_df = pd.DataFrame(updated_rows)

extraction_df.head()


## 7. Resumen de extracción

In [ ]:
print("Resumen por estado de extracción:")
display(extraction_df["status"].value_counts().reset_index().rename(columns={"index": "estado", "status": "cantidad"}))

print("Documentos con menor cantidad de caracteres extraídos:")
display(extraction_df.sort_values("characters").head(10))

print("Documentos con mayor cantidad de caracteres extraídos:")
display(extraction_df.sort_values("characters", ascending=False).head(10))


## 8. Revisión de documentos problemáticos

Se listan documentos que no pudieron extraerse correctamente o que tienen extracción parcial.


In [ ]:
problematic = extraction_df[extraction_df["status"].isin(["PDF_NO_ENCONTRADO", "SIN_TEXTO", "PARCIAL", "ERROR"])]

if problematic.empty:
    print("No se detectaron documentos problemáticos.")
else:
    print("Documentos a revisar:")
    display(problematic)


## 9. Guardado de reportes

Se guardan:

- reporte de extracción en CSV;
- corpus actualizado con información de extracción;
- reporte JSON resumen.


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

extraction_report_path = REPORTS_DIR / f"reporte_extraccion_texto_{timestamp}.csv"
updated_corpus_path = METADATA_DIR / "corpus_normativo_ambiental_con_extraccion.csv"
summary_json_path = REPORTS_DIR / f"resumen_extraccion_texto_{timestamp}.json"

extraction_df.to_csv(extraction_report_path, index=False, encoding="utf-8-sig")
updated_df.to_csv(updated_corpus_path, index=False, encoding="utf-8-sig")

summary = {
    "fecha_extraccion": datetime.now().isoformat(timespec="seconds"),
    "total_documentos": int(len(extraction_df)),
    "total_ok": int((extraction_df["status"] == "OK").sum()),
    "total_parcial": int((extraction_df["status"] == "PARCIAL").sum()),
    "total_sin_texto": int((extraction_df["status"] == "SIN_TEXTO").sum()),
    "total_error": int((extraction_df["status"] == "ERROR").sum()),
    "total_pdf_no_encontrado": int((extraction_df["status"] == "PDF_NO_ENCONTRADO").sum()),
    "total_caracteres": int(extraction_df["characters"].sum()),
    "total_palabras": int(extraction_df["words"].sum()),
}

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=4)

print(f"Reporte de extracción guardado en: {extraction_report_path}")
print(f"Corpus actualizado guardado en: {updated_corpus_path}")
print(f"Resumen JSON guardado en: {summary_json_path}")


## 10. Verificación de archivos `.txt` generados

In [ ]:
txt_files = sorted(PROCESSED_DIR.glob("*.txt"))

print(f"Archivos .txt generados: {len(txt_files)}")

for txt_file in txt_files[:10]:
    print(f"- {txt_file.name}")

if len(txt_files) > 10:
    print("...")


## 11. Resultado final

Si la mayoría de documentos quedó con estado `OK`, el corpus está listo para pasar al **Notebook 03 - Chunking estructural**.


In [ ]:
if (extraction_df["status"] == "OK").sum() > 0:
    print("Extracción completada. Revisa los reportes y los documentos problemáticos antes de continuar.")
else:
    print("No se logró extraer texto correctamente. Revisa los PDFs o considera OCR.")

display(extraction_df)
